# CRISP-DM: Fraud detection on `shop.db`

This notebook predicts **`is_fraud`** for rows in the **`orders`** table using SQLite data from the operational **shop** schema (IS 455 / Assignment 17.11 Part 2).

**Scope:** ML pipeline and serialized artifact. The Chapter 17 **web app** models **late delivery** and writes to **`order_predictions`** — that is a separate path from this fraud model.


## 1. Business Understanding

**Problem.** E-commerce orders are labeled as fraudulent (`is_fraud = 1`) or legitimate (`0`). We want a **binary classifier** that scores new orders so operations can review high-risk transactions.

**Success criteria.** Fraud is rare (~6% in this snapshot), so accuracy alone is misleading. We prioritize:
- **Average precision (PR-AUC)** — quality of ranking positives without fixing a threshold.
- **Recall + precision tradeoff** at an operating threshold tuned on validation data (catch fraud while controlling review workload).
- **ROC-AUC** as a secondary scalar summary.

**Constraints.** Single SQLite snapshot; no external data; deployment later must consume the same feature columns as training.

**Leakage note.** `risk_score` may be post-hoc or co-derived with fraud labels. The **principled** feature set **excludes** `risk_score`. A short comparison below shows how much metrics move if it were included (do not ship that variant without SME sign-off).


## 2. Data Understanding

Load **`orders`** with **`customers`**, plus per-order aggregates from **`order_items`**. We **do not** join **`shipments`** here: fulfillment outcomes are not assured at checkout time and can leak information unless the business explicitly scores orders after shipment exists.


In [1]:
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sqlite3

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.base import clone

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
PROJECT_ROOT = Path(".").resolve()
DB_PATH = PROJECT_ROOT / "db" / "shop.db"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

assert DB_PATH.exists(), f"Database not found: {DB_PATH}"
print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)

Project root: /Users/augusto.freire/CS Projects/IS 455/17.11 Deployment
Database: /Users/augusto.freire/CS Projects/IS 455/17.11 Deployment/db/shop.db


In [2]:
conn = sqlite3.connect(DB_PATH)

orders_sql = '''
SELECT
  o.order_id,
  o.customer_id,
  o.order_datetime,
  o.billing_zip,
  o.shipping_zip,
  o.shipping_state,
  o.payment_method,
  o.device_type,
  o.ip_country,
  o.promo_used,
  o.promo_code,
  o.order_subtotal,
  o.shipping_fee,
  o.tax_amount,
  o.order_total,
  o.risk_score,
  o.is_fraud,
  c.customer_segment,
  c.loyalty_tier,
  c.gender,
  c.city,
  c.state AS customer_state
FROM orders o
LEFT JOIN customers c ON c.customer_id = o.customer_id
'''

items_sql = '''
SELECT
  order_id,
  COUNT(*) AS line_count,
  SUM(quantity) AS sum_qty,
  COUNT(DISTINCT product_id) AS n_products
FROM order_items
GROUP BY order_id
'''

df = pd.read_sql(orders_sql, conn)
items_agg = pd.read_sql(items_sql, conn)
conn.close()

df = df.merge(items_agg, on="order_id", how="left")
for c in ["line_count", "sum_qty", "n_products"]:
    df[c] = df[c].fillna(0).astype(float)

df["order_datetime"] = pd.to_datetime(df["order_datetime"])
df["order_hour"] = df["order_datetime"].dt.hour.astype(float)
df["order_dow"] = df["order_datetime"].dt.dayofweek.astype(float)
df["billing_equals_shipping"] = (
    df["billing_zip"].fillna("").astype(str) == df["shipping_zip"].fillna("").astype(str)
).astype(int).astype(float)

print(df.shape)
df.head()

(5000, 28)


,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,promo_code,order_subtotal,shipping_fee,tax_amount,order_total,risk_score,is_fraud,customer_segment,loyalty_tier,gender,city,customer_state,line_count,sum_qty,n_products,order_hour,order_dow,billing_equals_shipping
0,1,1,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,None,662.95,15.44,46.30,724.69,38.3,0,standard,silver,Female,Clayton,CO,5.0,9.0,5.0,0.0,5.0,1.0
1,2,1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,SAVE10,862.92,14.74,66.61,944.27,94.9,0,standard,silver,Female,Clayton,CO,5.0,7.0,5.0,10.0,0.0,0.0
2,3,1,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,None,796.09,14.04,40.72,850.85,53.8,1,standard,silver,Female,Clayton,CO,3.0,5.0,3.0,7.0,0.0,1.0
3,4,1,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,WELCOME,137.60,6.99,11.88,156.47,4.2,0,standard,silver,Female,Clayton,CO,1.0,1.0,1.0,18.0,3.0,1.0
4,5,1,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,None,17.07,6.99,1.40,25.46,4.9,0,standard,silver,Female,Clayton,CO,1.0,1.0,1.0,5.0,6.0,1.0


In [3]:
TARGET = "is_fraud"
ID_COLS = ["order_id", "customer_id", "order_datetime"]
LEAKY_OPTIONAL = ["risk_score"]  # excluded from principled training; compare only

print("Class balance:")
print(df[TARGET].value_counts().rename("count"))
print(f"\nFraud rate: {df[TARGET].mean():.2%}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax[0], title="is_fraud counts")
ax[0].set_xticklabels(["legit (0)", "fraud (1)"], rotation=0)

numeric_profile = [
    "order_subtotal", "shipping_fee", "tax_amount", "order_total",
    "line_count", "sum_qty", "n_products",
]
for col in numeric_profile:
    df.groupby(TARGET)[col].mean().plot(kind="bar", ax=ax[1], alpha=0.6, label=col)
ax[1].set_title("Mean numeric features by class (overlap — see dist plots below)")
ax[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Class balance:
is_fraud
0    4682
1     318
Name: count, dtype: int64

Fraud rate: 6.36%


In [4]:
num_cols_eda = [
    "order_subtotal", "shipping_fee", "tax_amount", "order_total",
    "line_count", "order_hour", "order_dow",
]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()
for i, col in enumerate(num_cols_eda):
    ax = axes[i]
    sns.kdeplot(data=df, x=col, hue=TARGET, common_norm=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()
for ax, col in zip(axes, ["payment_method", "device_type", "ip_country", "customer_segment"]):
    ct = pd.crosstab(df[col].fillna("missing"), df[TARGET], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=ax, title=f"{col} vs fraud (row %)")
plt.tight_layout()
plt.show()

corr_cols = [c for c in num_cols_eda if c in df.columns]
plt.figure(figsize=(7, 5))
sns.heatmap(df[corr_cols + [TARGET]].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation (numeric + target)")
plt.tight_layout()
plt.show()

## 3. Data Preparation

**Stratified split** (60% / 20% / 20%) so each set keeps ~6% fraud. **Preprocessing:** median imputation + scaling for numeric features; constant `"unknown"` imputation + one-hot encoding for categoricals (infrequent levels collapsed via `max_categories` for high-cardinality zip fields).


In [5]:
def design_matrix(df_in: pd.DataFrame, include_risk_score: bool):
    y = df_in[TARGET].astype(int)
    drop_extra = list(ID_COLS)
    if not include_risk_score:
        drop_extra += [c for c in LEAKY_OPTIONAL if c in df_in.columns]
    X = df_in.drop(columns=[TARGET] + drop_extra)
    return X, y


X_principled, y = design_matrix(df, include_risk_score=False)
X_with_risk, y_chk = design_matrix(df, include_risk_score=True)
assert y.equals(y_chk)

cat_cols = [
    "billing_zip", "shipping_zip", "shipping_state",
    "payment_method", "device_type", "ip_country",
    "promo_code", "customer_segment", "loyalty_tier", "gender",
    "city", "customer_state",
]
num_cols = [
    "promo_used", "order_subtotal", "shipping_fee", "tax_amount", "order_total",
    "line_count", "sum_qty", "n_products",
    "order_hour", "order_dow", "billing_equals_shipping",
]

for c in cat_cols:
    X_principled[c] = X_principled[c].astype(str).where(X_principled[c].notna(), other=np.nan)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_principled, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
)

print("Train / val / test:", X_train.shape[0], X_val.shape[0], X_test.shape[0])
print("Fraud rates:", y_train.mean(), y_val.mean(), y_test.mean())

Train / val / test: 3000 1000 1000
Fraud rates: 0.06366666666666666 0.064 0.063


In [6]:
def build_preprocessor(cat_columns, num_columns) -> ColumnTransformer:
    numeric_tf = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_tf = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                max_categories=25,
                sparse_output=False,
            ),
        ),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_tf, num_columns),
            ("cat", categorical_tf, cat_columns),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


preprocessor = build_preprocessor(cat_cols, num_cols)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 4. Modeling

**Baselines and ensembles (Ch. 13–14):** `LogisticRegression` with `class_weight="balanced"`, **`RandomForestClassifier`**, and **`HistGradientBoostingClassifier`** (native handling of dense design matrix after OHE). **Cross-validation** uses **`StratifiedKFold`** and reports ROC-AUC and average precision.


In [7]:
def make_model_pipeline(classifier) -> Pipeline:
    return Pipeline([("prep", preprocessor), ("clf", classifier)])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "average_precision": "average_precision"}

candidates = {
    "logistic_regression": LogisticRegression(
        class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_depth=None,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_depth=6,
        max_iter=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

rows = []
fitted_base = {}
for name, est in candidates.items():
    pipe = make_model_pipeline(est)
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    row = {
        "model": name,
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
        "ap_mean": scores["test_average_precision"].mean(),
        "ap_std": scores["test_average_precision"].std(),
    }
    rows.append(row)
    # refit on full train for downstream
    pipe.fit(X_train, y_train)
    fitted_base[name] = pipe

cv_results = pd.DataFrame(rows).sort_values("ap_mean", ascending=False)
cv_results

,model,roc_auc_mean,roc_auc_std,ap_mean,ap_std
0,logistic_regression,0.695383,0.051315,0.172799,0.043207
2,hist_gradient_boosting,0.677709,0.019472,0.140457,0.026725
1,random_forest,0.683942,0.019915,0.137921,0.021898


## 5. Evaluation

Pick the best CV model by **mean average precision**. Tune a **probability threshold** on the **validation** set (here: maximize **F1**). Report **hold-out test** metrics and plots.


In [8]:
best_name = cv_results.iloc[0]["model"]
print("Selected:", best_name)
best_pipe = fitted_base[best_name]

val_probs = best_pipe.predict_proba(X_val)[:, 1]
threshold_grid = np.linspace(0.05, 0.95, 37)
best_threshold = 0.5
best_f1 = -1.0
for t in threshold_grid:
    preds = (val_probs >= t).astype(int)
    f1v = f1_score(y_val, preds, zero_division=0)
    if f1v > best_f1:
        best_f1 = f1v
        best_threshold = float(t)

print(f"Validation best F1≈{best_f1:.3f} at threshold: {best_threshold:.3f}")

test_probs = best_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= best_threshold).astype(int)

print("Hold-out ROC-AUC:", roc_auc_score(y_test, test_probs))
print("Hold-out average precision:", average_precision_score(y_test, test_probs))
print("Accuracy:", accuracy_score(y_test, test_pred))
print("\nClassification report (test):")
print(classification_report(y_test, test_pred, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, test_pred)
plt.title("Test confusion matrix")
plt.show()

fpr, tpr, _ = roc_curve(y_test, test_probs)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, test_probs):.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC curve (test)")
plt.legend()
plt.tight_layout()
plt.show()

prec_t, rec_t, _ = precision_recall_curve(y_test, test_probs)
plt.figure(figsize=(6, 4))
plt.plot(rec_t, prec_t, label=f"AP = {average_precision_score(y_test, test_probs):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–recall curve (test)")
plt.legend()
plt.tight_layout()
plt.show()

Selected: logistic_regression
Validation best F1≈0.228 at threshold: 0.475
Hold-out ROC-AUC: 0.700276126103234
Hold-out average precision: 0.15870855599654168
Accuracy: 0.722

Classification report (test):
              precision    recall  f1-score   support

           0      0.963     0.731     0.831       937
           1      0.128     0.587     0.210        63

    accuracy                          0.722      1000
   macro avg      0.546     0.659     0.521      1000
weighted avg      0.911     0.722     0.792      1000



In [9]:
# Optional: quantify how much `risk_score` inflates metrics if included (leakage risk).
preprocessor_risk = build_preprocessor(cat_cols, num_cols + ["risk_score"])
X_tr, X_te, y_tr, y_te = train_test_split(
    design_matrix(df, include_risk_score=True)[0],
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

def quick_ap(Xtr, ytr, Xte, yte):
    hgb = Pipeline([
        ("prep", preprocessor_risk),
        ("clf", HistGradientBoostingClassifier(
            learning_rate=0.06, max_depth=6, max_iter=150,
            class_weight="balanced", random_state=RANDOM_STATE,
        )),
    ])
    hgb.fit(Xtr, ytr)
    p = hgb.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, p), average_precision_score(yte, p)

auc_r, ap_r = quick_ap(X_tr, y_tr, X_te, y_te)
print("With risk_score on 80/20 split — ROC-AUC:", round(auc_r, 4), " AP:", round(ap_r, 4))
print("(Use principled feature set without risk_score for the saved artifact.)")

With risk_score on 80/20 split — ROC-AUC: 0.7321  AP: 0.1619
(Use principled feature set without risk_score for the saved artifact.)


## 6. Feature selection (Ch. 16)

**`SelectFromModel`** uses a `RandomForestClassifier` to estimate feature importances on the transformed space, then keeps supporters above the median importance. We refit the **same classifier family** (`HistGradientBoosting`) on reduced features and compare **validation** average precision vs. the full pipeline.


In [10]:
selector_estimator = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

prep_fitted = build_preprocessor(cat_cols, num_cols)
prep_fitted.fit(X_train, y_train)
X_train_t = prep_fitted.transform(X_train)
X_val_t = prep_fitted.transform(X_val)

sfm = SelectFromModel(selector_estimator, threshold="median")
sfm.fit(X_train_t, y_train)
n_before = X_train_t.shape[1]
n_after = int(np.sum(sfm.get_support()))
print(f"Transformed features: {n_before} -> {n_after} (via SelectFromModel median threshold)")

clf_reduced = HistGradientBoostingClassifier(
    learning_rate=0.06,
    max_depth=6,
    max_iter=200,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

clf_reduced.fit(sfm.transform(X_train_t), y_train)
val_probs_reduced = clf_reduced.predict_proba(sfm.transform(X_val_t))[:, 1]
ap_reduced = average_precision_score(y_val, val_probs_reduced)
ap_full_val = average_precision_score(y_val, val_probs)

comparison_fs = pd.DataFrame([
    {"variant": "full_pipe_val", "average_precision": ap_full_val},
    {"variant": "reduced_after_SFM_val", "average_precision": ap_reduced},
])
comparison_fs

Transformed features: 143 -> 72 (via SelectFromModel median threshold)


,variant,average_precision
0,full_pipe_val,0.137420
1,reduced_after_SFM_val,0.134066


In [11]:
# If selection hurts AP meaningfully, keep the full best pipeline; otherwise use reduced + HGB.
USE_REDUCED = bool(ap_reduced >= (ap_full_val - 0.005))
print("Use reduced feature pipeline for deployment?", USE_REDUCED)

Use reduced feature pipeline for deployment? True


## 7. Deployment (serialization)

**Chapter 17 in the course** often separates *training* from *serving*: the notebook trains once, persists **`joblib`**, and a batch script or API loads the artifact. This fraud model is **not** the same as the late-delivery **`order_predictions`** job, but the pattern is identical: **load model → compute `predict_proba` → persist scores** (here we only demonstrate load + predict).

Refit on **train + validation** (hold test untouched for the report above), package preprocessor + classifier (full or reduced), save **`fraud_model.joblib`** and **`fraud_model_metadata.json`** under **`artifacts/`**.

For the reduced pipeline, `predict_proba` comes from the final `HistGradientBoostingClassifier`; `predict` uses that estimator’s default **0.5** cutoff. To apply the **tuned threshold** from metadata, use `(predict_proba(X)[:, 1] >= threshold).astype(int)`.


In [12]:
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

if USE_REDUCED:
    # sklearn Pipeline is picklable (avoids local class joblib issues)
    final_model = Pipeline(
        [
            ("prep", build_preprocessor(cat_cols, num_cols)),
            (
                "select",
                SelectFromModel(
                    RandomForestClassifier(
                        n_estimators=200,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                    threshold="median",
                ),
            ),
            (
                "clf",
                HistGradientBoostingClassifier(
                    learning_rate=0.06,
                    max_depth=6,
                    max_iter=200,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    final_model.fit(X_trainval, y_trainval)
    model_kind = "reduced_sfm_plus_hgb"
else:
    final_estimator = clone(candidates[best_name])
    prep_final = build_preprocessor(cat_cols, num_cols)
    final_model = Pipeline([("prep", prep_final), ("clf", final_estimator)])
    final_model.fit(X_trainval, y_trainval)
    model_kind = f"full_pipeline_{best_name}"

MODEL_PATH = ARTIFACT_DIR / "fraud_model.joblib"
META_PATH = ARTIFACT_DIR / "fraud_model_metadata.json"

joblib.dump(final_model, MODEL_PATH)

test_ap_full = average_precision_score(y_test, final_model.predict_proba(X_test)[:, 1])

meta = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "target": TARGET,
    "db_path": str(DB_PATH),
    "model_kind": model_kind,
    "best_cv_model": best_name,
    "threshold": best_threshold,
    "n_features_raw_cat": len(cat_cols),
    "n_features_raw_num": len(num_cols),
    "principled_excludes": LEAKY_OPTIONAL,
    "holdout_test_average_precision_after_refit_trainval": float(test_ap_full),
}

with META_PATH.open("w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved:", MODEL_PATH)
print("Saved:", META_PATH)
meta

Saved: /Users/augusto.freire/CS Projects/IS 455/17.11 Deployment/artifacts/fraud_model.joblib
Saved: /Users/augusto.freire/CS Projects/IS 455/17.11 Deployment/artifacts/fraud_model_metadata.json


{'created_at_utc': '2026-03-31T16:17:09.964480+00:00',
 'target': 'is_fraud',
 'db_path': '/Users/augusto.freire/CS Projects/IS 455/17.11 Deployment/db/shop.db',
 'model_kind': 'reduced_sfm_plus_hgb',
 'best_cv_model': 'logistic_regression',
 'threshold': 0.475,
 'n_features_raw_cat': 12,
 'n_features_raw_num': 11,
 'principled_excludes': ['risk_score'],
 'holdout_test_average_precision_after_refit_trainval': 0.11238627308128761}

In [13]:
# Load + score a few test rows (integration smoke test)
loaded = joblib.load(MODEL_PATH)
sample_X = X_test.head(5)
probs = loaded.predict_proba(sample_X)[:, 1]
pd.DataFrame({"fraud_probability": probs}).assign(true_label=y_test.head(5).values)

,fraud_probability,true_label
0,0.024991,0
1,0.031332,0
2,0.532190,0
3,0.052932,0
4,0.004659,0
